In [1]:
# 导包
import torch
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torchsummary import summary
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
from torchvision import  transforms
import matplotlib.pylab as plt
import time
BATCH_SIZE = 128

In [2]:
# 数据集的基本分析
def create_data():
    # 对数据集进行加强
    # CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
    # CIFAR_STD = (0.2023, 0.1994, 0.2010)

    # train_transform = transforms.Compose([
    #     transforms.RandomCrop(32, padding=4),
    #     transforms.RandomHorizontalFlip(),
    #     transforms.ToTensor(),
    #     transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    # ])

    # eval_transform = transforms.Compose([
    #     transforms.ToTensor(),
    #     transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    # ])
    # 加载数据集
    train_dataset = CIFAR10(root="../data",train=True,transform=ToTensor(),download=True)
    test_dataset = CIFAR10(root="../data",train=False,transform=ToTensor(),download=True)
    return train_dataset,test_dataset


In [3]:
# 构建CNN类/网络模型 这是保证测试数据准确率的关键
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 定义网络层
        # 第一层
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.maxpool1 = nn.MaxPool2d(2,2)
        # 批量归一化 对长和宽进行归一化
        self.dn1 = nn.BatchNorm2d(32)
        nn.init.kaiming_normal_(self.conv1.weight,nonlinearity='relu')
        nn.init.zeros_(self.conv1.bias)
        # 第二层
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.maxpool2 = nn.MaxPool2d(2,2)
        self.dn2 = nn.BatchNorm2d(64)
        nn.init.kaiming_normal_(self.conv2.weight,nonlinearity='relu')
        nn.init.zeros_(self.conv2.bias)
        # # 第三层
        # self.conv3 = nn.Conv2d(16,32,3,padding=1)
        # self.maxpool3 = nn.MaxPool2d(2,2)
        # self.dn3 = nn.BatchNorm2d(32)
        # nn.init.kaiming_normal_(self.conv3.weight,nonlinearity='relu')
        # nn.init.zeros_(self.conv3.bias)
        # # 第四层
        # self.conv4 = nn.Conv2d(32,64,3,padding=1)
        # self.maxpool4 = nn.MaxPool2d(2,2)
        # self.dn4 = nn.BatchNorm2d(64)
        # nn.init.kaiming_normal_(self.conv4.weight,nonlinearity='relu')
        # nn.init.zeros_(self.conv4.bias)
        # 全连接层 (可以用随机失活 p可以弄大一点)
        self.linear1 = nn.Linear(4096,120)
        # 对输出的进行归一化 一维
        self.ln1 = nn.BatchNorm1d(120)
        nn.init.kaiming_normal_(self.linear1.weight,nonlinearity='relu')
        nn.init.zeros_(self.linear1.bias)
        self.droup1 = nn.Dropout(p=0.5)
        self.linear2 = nn.Linear(120,84)
        self.ln2 = nn.BatchNorm1d(84)
        nn.init.kaiming_normal_(self.linear2.weight,nonlinearity='relu')
        nn.init.zeros_(self.linear2.bias)
        self.droup2 = nn.Dropout(p=0.5)
        self.out = nn.Linear(84,10)
    def forward(self,x):
        # 向前传播
        x = self.maxpool1(torch.relu(self.dn1(self.conv1(x))))
        x = self.maxpool2(torch.relu(self.dn2(self.conv2(x))))
        # x = self.maxpool3(torch.relu(self.dn3(self.conv3(x))))
        # x = self.maxpool4(torch.relu(self.dn4(self.conv4(x))))
        # 将图像转换为特征向量 准备做全链接
        x = x.view(x.size(0),-1)
        x = torch.relu(self.ln1(self.linear1(x)))
        x = self.droup1(x)
        x = torch.relu(self.ln2(self.linear2(x)))
        x = self.droup2(x)
        return self.out(x)



In [4]:
# 模型训练
def train(train_dataset):
    # 数据加载
    dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    # 创建模型对象
    model = CNN()
    # 创建损失函数
    loss_fn = nn.CrossEntropyLoss()
    # 创建优化器 后面的参数是科学计数法
    optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)
    # 定义循环次数
    epochs = 15
    # 循环训练函数
    # 开始所有的轮次
    for epoch in range(epochs):
        total_loss = 0.0
        batch_num = 0
        start = time.time()
        corrent_num = 0
        sample_num = 0
        for x,y in dataloader:
            # print(f"x的形状：{x.shape},x的类型：{x.dtype}")
            # print(f"y的形状：{y.shape},y的类型：{y.dtype}")
            # 向前传播
            y_pred = model(x)
            # 梯度清零
            optimizer.zero_grad()
            # 计算损失
            loss = loss_fn(y_pred,y)
             # 反向传播
            loss.backward()
            total_loss += loss.item() * len(y)
            # 更新权重
            optimizer.step()
            # 得到预测值中类别最大的索引值是什么
            y_pred = torch.argmax(y_pred,dim=-1)

            # print(f"y_pred:\n{y_pred}")
            # print(f"y_ture:\n{y}")
            sample_num += len(y)
            # 计算正确的数量
            corrent_num += (y_pred == y).sum().item()

            # break
        print(f"epoch:{epoch+1},Loss:{total_loss/sample_num},正确率:{corrent_num/sample_num},time:{time.time()-start}")
    # 训练完成 保存模型
    torch.save(model.state_dict(),"img_select.pkl")
    print("训练完成")


In [5]:
# 模型评估
def evaluate(test_dataset):
    # 创建数据加载器
    dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    # 创建模型
    model = CNN()
    # 加载模型
    model.load_state_dict(torch.load("img_select.pkl"))
    # 测试评估
    total_num = 0
    total_current = 0
    for x,y in dataloader:
        # 修改模型状态为预测模型
        model.eval()
        # 模型预测
        y_pred = model(x)
        # softmax激活 将模型的评分转换为概率
        y_pred = torch.softmax(y_pred,dim=-1)
        # 返回概率最大类别索引
        y_pred = torch.argmax(y_pred, dim=-1)
        # 计算正确的数量
        total_current += (y_pred == y).sum().item()
        # 计算总的数量
        total_num += len(y)
    # 计算完毕 打印正确的结果
    print(f"训练集的长度：{total_num}")
    print(f"预测正确的数量：{total_current}")
    print(f"正确率：{total_current/total_num * 100}")

In [6]:
if __name__ == "__main__":
    train_dataset,test_dataset = create_data()
    print(f"数据集类别:{train_dataset.class_to_idx}")
    print(f"训练集规模:{train_dataset.data.shape}")
    print(f"测试集规模:{test_dataset.data.shape}")
    # 图像展示  
    # plt.figure(figsize=(2, 2))   
    # plt.imshow(train_dataset.data[10086])   
    # plt.title(train_dataset.targets[1])    
    # plt.show()
    model = CNN()
    # 输入的通道数必需是3
    summary(model,input_size=(3,32,32))
    # train(train_dataset)
    # evaluate(test_dataset)

数据集类别:{'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}
训练集规模:(50000, 32, 32, 3)
测试集规模:(10000, 32, 32, 3)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]             896
       BatchNorm2d-2           [-1, 32, 32, 32]              64
         MaxPool2d-3           [-1, 32, 16, 16]               0
            Conv2d-4           [-1, 64, 16, 16]          18,496
       BatchNorm2d-5           [-1, 64, 16, 16]             128
         MaxPool2d-6             [-1, 64, 8, 8]               0
            Linear-7                  [-1, 120]         491,640
       BatchNorm1d-8                  [-1, 120]             240
           Dropout-9                  [-1, 120]               0
           Linear-10                   [-1, 84]          10,164
      BatchNorm1d-11                   [-1, 84]       